# `001 / R-GCN.DistMult`

In [1]:
COMMIT = "49ae806c37f1e992fefd7d686afa90d247c7b38d"
DATA_REV = "a111a7e06387ddb5943684dc1d7f240b1683f51f"
MANIFEST = "4536f73cd790cb64f4c7614c243af3c82c0a2e978ade032049c51af75221895e"

PREFLIGHT = ""
LAUNCH = True
RESUME = False
STOP_AFTER = None

FIT = [
    "--epochs", "500", "--width", "128",
    "--learning-rate", "0.0005", "--negatives", "8",
    "--eval-every", "5", "--patience", "15",
    "--weight-decay", "0.0001", "--min-epochs", "50",
    "--map-min-delta", "0.0001",
]

In [2]:
import hashlib
import os
import re
import subprocess
import sys
from pathlib import Path

from google.colab import userdata


def run(*args, cwd=None):
    command = [str(arg) for arg in args]
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}

    print("$", " ".join(command), flush=True)

    with subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()

    if status != 0:
        raise subprocess.CalledProcessError(status, command)


for description, value, length in (
    ("source commit", COMMIT, 40),
    ("dataset revision", DATA_REV, 40),
    ("manifest checksum", MANIFEST, 64),
):
    if re.fullmatch(rf"[0-9a-f]{{{length}}}", value) is None:
        raise ValueError(f"{description} must have {length} lowercase hex digits")

token = userdata.get("exapt_hf")
if not token:
    raise RuntimeError("hugging face token missing from colab secrets")
os.environ["HF_TOKEN"] = token
del token

repo = Path("/content/exapt")
if not repo.exists():
    run("git", "clone", "https://github.com/aekn/exapt.git", repo)
if subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=repo, text=True
).strip():
    raise RuntimeError("source checkout has uncommitted changes")
run("git", "checkout", "--detach", COMMIT, cwd=repo)
if subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repo, text=True
).strip() != COMMIT:
    raise RuntimeError("source checkout does not match the pinned commit")

run(sys.executable, "-m", "pip", "install", "-q", "uv")
run("uv", "sync", "--locked", "--python", "3.12", cwd=repo)
run("uv", "run", "python", "-c", "import torch; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print(torch.__version__, torch.version.cuda, "
    "torch.cuda.get_device_name(0))", cwd=repo)

prepared = repo / "data/processed"
run("uv", "run", "exapt", "hub", "download",
    "exapt-lab/data", DATA_REV, prepared, cwd=repo)
actual = hashlib.sha256((prepared / "manifest.json").read_bytes()).hexdigest()
if actual != MANIFEST:
    raise RuntimeError("dataset manifest checksum does not match")
run("uv", "run", "python", "-m", "unittest", "discover",
    "-s", "tests", "-q", cwd=repo)

$ git clone https://github.com/aekn/exapt.git /content/exapt
Cloning into '/content/exapt'...
$ git checkout --detach 49ae806c37f1e992fefd7d686afa90d247c7b38d
HEAD is now at 49ae806 turn off them hf progress bars
$ /usr/bin/python3 -m pip install -q uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 11.0 MB/s eta 0:00:00
$ uv sync --locked --python 3.12
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 44 packages in 1ms
Prepared 42 packages in 1m 56s
Installed 42 packages in 216ms
 + anyio==4.15.1
 + click==8.5.0
 + cuda-bindings==13.4.4
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + exapt==0.1.0 (from file:///content/exapt)
 + filelock==4.2.4
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.7.0
 + httpcore2==2.16.0
 + httpx2==2.16.0
 + huggingface-hub==2.2.0
 + idna==3.20
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-

In [3]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

if PREFLIGHT not in ("", "overfit", "profile"):
    raise ValueError(
        f"unknown preflight mode {PREFLIGHT!r}; use overfit or profile"
    )
if PREFLIGHT == "overfit":
    run("uv", "run", "exapt", "preflight", "overfit",
        prepared / "split/smoke", "--width", "32",
        "--steps", "300", "--device", "cuda", cwd=repo)
elif PREFLIGHT == "profile":
    run("uv", "run", "exapt", "preflight", "profile-fit",
        prepared / "graph.pt", prepared / "split",
        repo / "runs/preflight-001-fit", "--width", "128",
        "--device", "cuda", cwd=repo)
elif not LAUNCH:
    print("Ready: 001 / fit. Training disabled.")
else:
    output = repo / "runs/001/fit"
    flags = []
    if RESUME:
        if not (output / "latest.json").is_file():
            run("uv", "run", "exapt", "hub", "recover",
                "exapt-lab/hangar", "001", "fit", output, cwd=repo)
        flags += ["--resume", str(output / "latest.json")]
    if STOP_AFTER is not None:
        flags += ["--stop-after", str(STOP_AFTER)]

    print("Run: 001 / fit")
    run("uv", "run", "exapt", "train", prepared / "graph.pt",
        prepared / "split", output, "--device", "cuda",
        "--run-id", "001", "--bucket", "exapt-lab/hangar",
        "--source-commit", COMMIT, "--dataset-revision", DATA_REV,
        "--manifest-sha256", MANIFEST, *FIT, *flags, cwd=repo)

Run: 001 / fit
$ uv run exapt train /content/exapt/data/processed/graph.pt /content/exapt/data/processed/split /content/exapt/runs/001/fit --device cuda --run-id 001 --bucket exapt-lab/hangar --source-commit 49ae806c37f1e992fefd7d686afa90d247c7b38d --dataset-revision a111a7e06387ddb5943684dc1d7f240b1683f51f --manifest-sha256 4536f73cd790cb64f4c7614c243af3c82c0a2e978ade032049c51af75221895e --epochs 500 --width 128 --learning-rate 0.0005 --negatives 8 --eval-every 5 --patience 15 --weight-decay 0.0001 --min-epochs 50 --map-min-delta 0.0001
/content/exapt/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
/content/exapt/.venv/lib/python3.12/site-packages/torch/serialization.py:2244: UserWarning: Validating sparse tensor invariants because weights_only